# R6 따라 하기: 대조군을 덜 읽으면 무엇이 달라지나 (재학습 없음)

이 노트북은 두 부분입니다.

1. **장난감 예시(가상 숫자)** 로 count 박화와 library size 계산을 손으로 확인합니다. 실제 세포 자료가 필요 없습니다.
2. 저장소에 커밋된 **작은 결과표(실제 수치)** 를 읽어 관측 진단 → 개발 fold 비교 → 동결 뒤 시험 약물 평가를 다시 봅니다.

필요한 것: numpy, pandas, matplotlib. GPU·서버·세포 배열은 필요 없습니다. 저장소 루트에서 실행하세요.

`jupyter nbconvert --to notebook --execute --inplace tutorials/depth_robust_r6/01_thinning_and_results.ipynb`

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "reports").exists() else Path.cwd().parents[1]
T = ROOT / "reports/depth_robust_r6/tables"
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print(sorted(p.name for p in T.iterdir()))

## 1. 장난감 예시 (가상 숫자): 한 세포를 25%만 읽기

세포 하나는 유전자별 UMI 개수입니다. "25%만 읽는다"는 것은 **분자 하나하나를 확률 0.25로 남기는 것**입니다(`Binomial(c, 0.25)`).
panel 밖 유전자의 합도 같은 방식으로 줄이고, library size는 **줄어든 값들의 합**으로 다시 계산합니다. 정규화 식은 항상 `log1p(1e4 · c / library)` 입니다.

In [ ]:
rng = np.random.default_rng(0)
c = np.array([40, 12, 5, 2, 1, 1, 0, 0])      # 가상: panel 유전자 8개의 UMI
c_rest = 900                                   # 가상: panel 밖 유전자의 UMI 합
def view(c, c_rest, p, rng):
    if p == 1.0:
        return c.copy(), c_rest               # p=1은 아무것도 뽑지 않고 그대로
    return rng.binomial(c, p), int(rng.binomial(c_rest, p))
rows = []
for p in (1.0, 0.5, 0.25):
    cp, rp = view(c, c_rest, p, rng)
    lib = cp.sum() + rp                        # 박화된 합 = 새 library size
    rows.append(dict(p=p, counts=cp.tolist(), library=int(lib), detected=int((cp > 0).sum()),
                     lognorm=np.round(np.log1p(1e4 * cp / lib), 2).tolist()))
print(pd.DataFrame(rows).to_string(index=False))

읽을 점: (i) 1개짜리 유전자는 자주 0이 됩니다(검출 유전자 수가 줄어듦). (ii) 남은 유전자의 정규화 값은 깊이가 얕아질수록 **들쭉날쭉**해집니다.
(iii) 그래서 세포 64개의 평균 프로필도 깊이에 따라 체계적으로 이동합니다. 아래 실제 수치에서 그 크기를 봅니다.

박화한 세포는 **같은 세포를 덜 읽은 것**입니다. 새 세포나 독립 반복이 아닙니다.

## 2. 실제 수치: 관측 진단 (R6-0)

In [ ]:
o = json.loads((T / "r6_0_observation.json").read_text())
v = pd.DataFrame(o["views"])
keep = ["view", "median_library", "median_detected_panel_genes", "frac_cells_with_fewer_than_256_tokens",
        "setmean_shift_rms_in_between_pool_sd", "nearest_pool_correct_centred", "nearest_pool_correct_raw_euclid", "top256_token_overlap_with_full_mean"]
print(v[keep].round(3).to_string(index=False))
print("pools:", o["n_pools"], " truth delta RMS:", round(o["truth_delta_rms"], 4))

`setmean_shift…`는 "대조군 64개 평균 프로필이 전체 관측 대비 얼마나 움직였나"를 pool 사이 표준편차 단위로 잰 값입니다. p=0.25에서는 pool 사이 차이만큼 움직입니다.
그래도 평균 이동을 빼면(`centred`) 어느 pool인지는 100% 맞힙니다. 즉 **정보는 남아 있고, 척도가 이동**합니다.

## 3. 실제 수치: 개발 fold — head 수정 전후와 arm 비교

개발 fold는 train 약물 안의 79개입니다. 설정을 고르는 데 쓴 값이므로 일반화 추정치가 아닙니다.

In [ ]:
r = pd.read_csv(T / "pilot_runs.tsv", sep="\t")
r = r[r.status == "ok"]
best = r.loc[r.groupby(["arm", "head", "drug_shrink"]).dev_p025.idxmin()]
print(best[["arm", "head", "drug_shrink", "lr", "dev_p1", "dev_p05", "dev_p025"]].sort_values(["head", "drug_shrink", "arm"]).round(4).to_string(index=False))

`joint` = 처음 head, `additive` = 수정 1(상태 전용 항 + 눌러 둔 약물 특이 항). 수정 1은 모든 arm에 똑같이 적용한 기본 모델 변경입니다.

`TEACH0/TEACH1`은 박화 증강 없이 전체 관측만 본 모델입니다. p=0.25 입력에서만 크게 나빠집니다 → **증강이 깊이 문제를 거의 다 해결**합니다.

## 4. 실제 수치: 동결 뒤 시험 약물 70개

In [ ]:
m = pd.read_csv(T / "final_main.tsv", sep="\t")
w = m.pivot(index="arm", columns="p", values="delta_rmse").loc[lambda d: d[0.25].sort_values().index]
print(w.round(4).to_string())
p = pd.read_csv(T / "final_pairs.tsv", sep="\t")
print(p[p.p.isin([0.25, 1.0])][["a", "b", "p", "diff", "lo", "hi", "rel", "better_by_rule", "worse_by_rule"]].round(5).to_string(index=False))

`better_by_rule`은 '구간이 0을 제외하고 상대 차이가 1% 이상'일 때만 True입니다. 구간은 약물 그룹 70개를 다시 뽑아 계산했습니다(세포·seed는 독립 표본이 아님).

In [ ]:
import matplotlib.pyplot as plt
order = [a for a in ["linemean", "Z0", "B00", "B10", "B01", "B11", "DEN", "TEACH0"] if a in w.index]
fig, ax = plt.subplots(figsize=(7, 4))
for a in order:
    ax.plot([0, 1, 2], w.loc[a, [0.25, 0.5, 1.0]], marker="o", lw=2, label=a, ls="--" if a in ("linemean", "TEACH0") else "-")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["p=0.25", "p=0.5", "p=1.0"]); ax.set_ylabel("delta RMSE (lower is better)")
ax.legend(ncol=4, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12)); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); plt.show()

## 5. SNU-423 (사전 지정한 HCC 하위 표)

세포주 하나의 결과입니다. 환자 HCC로 일반화하지 않습니다.

In [ ]:
s = pd.read_csv(T / "final_snu423.tsv", sep="\t")
print(s[(s.kind == "arm") & s.p.isin([0.25, 1.0])][["a", "p", "value", "lo", "hi", "n_drugs"]].round(4).to_string(index=False))
print(s[(s.kind == "pair") & (s.p == 0.25)][["a", "b", "value", "lo", "hi"]].round(5).to_string(index=False))

## 정리

- 입력: 대조군 세포 64개의 (덜 읽은) UMI + 약물 구조. 정답: 실제로 측정한 처리 후 평균 변화(전체 관측, p와 무관).
- 무엇을 숨겼나: 시험 약물 70개의 처리군 전체. 세포주와 plate는 학습에서 본 것입니다.
- 결론과 한계는 `reports/depth_robust_r6/RESULTS.md`, `CLAIMS_AND_LIMITATIONS.md`를 보세요.